# GRPO on `qwen2-vl-scicap-lora-adapter-dim_with_textlm`

Runs GRPO (TRL + Unsloth) starting from the **SFT adapter**
`shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm`, and pushes the
RL-tuned adapter to a **new** Hub repo.

**What this notebook keeps consistent with how that adapter was trained**

| Thing | Value | Why |
|---|---|---|
| Base | `Qwen/Qwen2.5-VL-7B-Instruct` (4-bit) | resolved automatically from the adapter's `adapter_config.json` |
| Pixel budget | `size = {shortest_edge: 256*28*28, longest_edge: 1280*28*28}` | SFT wired this into the processor; if you skip it the model sees a different token budget than it trained on |
| Prompt | `Regions: [{bbox_2d, caption}] ... Summary: ...` (+ optional `Context read from the figure:` prefix) | must match the SFT prompt or GRPO is optimizing an off-distribution policy |
| Completion length | 256 | region JSON + summary does not fit in 128 |

**Reward = CLIP cycle-consistency on the `Summary:` span + a small format bonus.**
Scoring the *whole* completion with CLIP would be wrong here: the raw JSON
region list is not a caption, so CLIP similarity on it is noise. This notebook
extracts the text after `Summary:` and scores that, and adds a small shaping
term for well-formed region JSON.

**Order of cells:** run 1 -> 2 (restart) -> 3 onward.

---
## 1. Install

In [ ]:
!pip install --quiet --upgrade pip
!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --upgrade unsloth_zoo "trl>=0.20.0"
!pip install --quiet "bitsandbytes>=0.46.0"
!pip install --quiet kaggle evaluate rouge_score sacrebleu
print("Installed.  Now:  Runtime > Restart session,  then continue from Cell 3.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Installed.  Now:  Runtime > Restart session,  then continue from Cell 3.


Optional hard reset if a previous session left a stale compiled cache:

In [ ]:
import os, shutil
if os.path.exists("/content/.deps_installed"): os.remove("/content/.deps_installed")
shutil.rmtree("/content/unsloth_compiled_cache", ignore_errors=True)
print("cache cleared")

---
## 2. Knobs

Everything you would normally want to change lives here.

In [ ]:
# ---- source / destination -------------------------------------------------
SFT_ADAPTER = "shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm"   # start point
GRPO_REPO   = "shemalfoy/qwen2-vl-scicap-grpo-dim_with_textlm"           # NEW repo (do not overwrite the SFT one)
PUSH_EVERY  = 50          # push adapter to Hub every N optimizer steps

# ---- data slice -----------------------------------------------------------
# The SFT adapter was trained on rows 0-1999. Starting GRPO at 2000 keeps the
# RL phase on figures the policy has never been supervised on. Set to 0 if you
# deliberately want to RL on the same slice.
GRPO_START  = 2000
GRPO_N      = 500         # number of figures for GRPO
MAX_STEPS   = 255         # -1 => full epoch. 255 matches the earlier GRPO run.

# ---- Feature 1: text-LM context hint --------------------------------------
# The SFT adapter saw "Context read from the figure: ..." on figures where a
# caption paragraph was detected. Turning this ON reproduces that prompt shape
# but costs a Qwen2.5-7B pass over the slice (~20-40 min for 500 figs on a T4/L4).
# OFF is still in-distribution -- SFT also saw plenty of context-free prompts.
USE_CONTEXT = False

# ---- image sizing before generation ---------------------------------------
# GRPO samples num_generations completions per figure, so image tokens dominate
# VRAM. Pre-shrinking the longest side to 1024 keeps small-text legibility while
# staying well inside the 1280*28*28 ceiling.
GRPO_LONGEST = 1024

print("start:", SFT_ADAPTER, "\n  ->  ", GRPO_REPO)

start: shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm 
  ->   shemalfoy/qwen2-vl-scicap-grpo-dim_with_textlm


---
## 3. Load the SFT adapter (base is resolved automatically)

`FastVisionModel.from_pretrained` on an adapter repo pulls
`Qwen/Qwen2.5-VL-7B-Instruct` in 4-bit and applies the LoRA weights on top, so
there is no separate merge step. `fast_inference=False` avoids the vLLM
LoRA+vision limitation.

In [ ]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

from unsloth import FastVisionModel, is_bf16_supported
import torch

SFT_ADAPTER = globals().get("SFT_ADAPTER", "shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm")

model, tokenizer = FastVisionModel.from_pretrained(
    SFT_ADAPTER,
    max_seq_length = 16384,     # large enough to fit image tokens
    load_in_4bit   = True,
    fast_inference = False,
)

# ---- re-wire the SAME pixel budget the adapter was fine-tuned under --------
min_pixels = 256  * 28 * 28
max_pixels = 1280 * 28 * 28

def set_pixel_budget(proc, mn, mx):
    # min_pixels/max_pixels are read-only properties in current transformers;
    # image_processor.size is the real source of truth that smart_resize reads.
    ip = getattr(proc, "image_processor", proc)
    try:
        ip.size = {"shortest_edge": mn, "longest_edge": mx}
    except Exception as e:
        print("size set failed:", e)
    for obj in (ip, proc):
        for attr, val in (("min_pixels", mn), ("max_pixels", mx)):
            try:
                setattr(obj, attr, val)
            except (AttributeError, TypeError):
                pass
    return ip

_ip = set_pixel_budget(tokenizer, min_pixels, max_pixels)
print("pixel budget wired ->", _ip.size)

FastVisionModel.for_training(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")
assert trainable > 0, "No trainable params -- the adapter did not load as PEFT."

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.21: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Unsloth: Offloading embeddings to RAM to save 1.02 GB.
pixel budget wired -> {'shortest_edge': 200704, 'longest_edge': 1003520}
trainable 51,521,536 / 5,081,043,968  (1.01%)


---
## 4. SciCap from Kaggle

Same source the SFT run used: the `dhivyaraman123/scicap-dataset` Kaggle
dataset. Upload your `kaggle.json` (Kaggle > Settings > API > Create New Token).

In [ ]:
from google.colab import files
print("Select your kaggle.json file.")
files.upload()
print("kaggle.json uploaded.")

Select your kaggle.json file.


Saving kaggle (1).json to kaggle (1).json
kaggle.json uploaded.


In [ ]:
import os

if not os.path.exists('kaggle (1).json'):
    print("WARNING: kaggle.json not found. Run the cell above first.")
else:
    os.environ['KAGGLE_CONFIG_DIR'] = os.getcwd()
    if not os.path.exists('/content/scicap_data'):
        print("Downloading SciCap from Kaggle ...")
        !kaggle datasets download -d dhivyaraman123/scicap-dataset
        !unzip -q scicap-dataset.zip -d scicap_data
    else:
        print("scicap_data already present -- skipping download.")

Dataset URL: https://www.kaggle.com/datasets/dhivyaraman123/scicap-dataset
License(s): CC0-1.0
100% 19.1G/19.1G [15:19<00:00, 22.3MB/s]



In [ ]:
import os
import pandas as pd
from datasets import Dataset, Image

csv_path  = '/content/scicap_data/scicap_train.csv'
image_dir = '/content/scicap_data/scicap_images_compressed/share-task-img-mask/arxiv/train'

df = pd.read_csv(csv_path)
df['image'] = df['image'].astype(str).str.strip().map(os.path.basename)

# Vectorized existence check: one directory listing instead of 300k stat calls
existing = set(os.listdir(image_dir))
df = df[df['image'].isin(existing)].copy()
df['image_path'] = image_dir + os.sep + df['image']

print(f"Found {len(df)} valid image-caption pairs")

full_ds = Dataset.from_dict({
    'image':   df['image_path'].tolist(),
    'caption': df['caption'].astype(str).tolist(),
})
full_ds = full_ds.cast_column('image', Image())   # lazy decode
print(full_ds)

Found 333472 valid image-caption pairs
Dataset({
    features: ['image', 'caption'],
    num_rows: 333472
})


---
## 5. Build the GRPO slice

Two things happen here:

1. **Images are pre-shrunk to disk** (longest side `GRPO_LONGEST`) and the column
   is re-cast as an `Image` feature. This is deliberate rather than using
   `set_transform` -- a live transform masks the `prompt` column and GRPO then
   errors out with a missing key.
2. The `prompt` column is built in the **exact SFT format**.

In [ ]:
from PIL import Image as PILImage
from datasets import Image as HFImage
import os, json

WORK = "/content/grpo_imgs"
os.makedirs(WORK, exist_ok=True)

full_ds.reset_format()          # make sure no transform is masking columns
end = min(GRPO_START + GRPO_N, len(full_ds))
grpo_raw = full_ds.select(range(GRPO_START, end))
print(f"GRPO slice: rows {GRPO_START}..{end-1}  ({len(grpo_raw)} figures)")

def _shrink(ex, idx):
    img = ex["image"].convert("RGB")
    w, h = img.size
    s = GRPO_LONGEST / max(w, h)
    if s < 1.0:
        img = img.resize((max(1, int(w*s)), max(1, int(h*s))))
    p = f"{WORK}/{idx:06d}.jpg"
    img.save(p, quality=92)
    return {"image": p, "caption": ex["caption"]}

grpo_raw = grpo_raw.map(_shrink, with_indices=True,
                        remove_columns=grpo_raw.column_names)
grpo_raw = grpo_raw.cast_column("image", HFImage())
print("images materialized ->", WORK)

GRPO slice: rows 2000..2499  (500 figures)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

images materialized -> /content/grpo_imgs


In [ ]:
# Optional Feature-1 context. Skipped entirely when USE_CONTEXT is False.
if USE_CONTEXT:
    !pip install -q easyocr
    import easyocr, numpy as np, torch
    from transformers import pipeline, BitsAndBytesConfig

    _det = easyocr.Reader(['en'], gpu=True)

    def detect_text_block(pil_img, bottom_frac=0.55, min_lines=2):
        """Locate a stacked block of text lines in the lower part of the figure.
        Detector only -- geometry, nothing is read here."""
        arr = np.asarray(pil_img.convert("RGB"))
        H, W = arr.shape[:2]
        horizontal_list, _ = _det.detect(arr)
        boxes = horizontal_list[0] if horizontal_list else []
        lines = []
        for x_min, x_max, y_min, y_max in boxes:
            cy = (y_min + y_max) / 2
            if cy < H * (1 - bottom_frac):      # not in the lower band
                continue
            if (x_max - x_min) < W * 0.12:      # too short to be prose
                continue
            lines.append((x_min, y_min, x_max, y_max))
        if len(lines) < min_lines:
            return False, None
        x1 = min(l[0] for l in lines); y1 = min(l[1] for l in lines)
        x2 = max(l[2] for l in lines); y2 = max(l[3] for l in lines)
        return True, [round(x1/W*1000), round(y1/H*1000),
                      round(x2/W*1000), round(y2/H*1000)]

    def _crop_norm(pil_img, box):
        W, H = pil_img.size; x1, y1, x2, y2 = box
        return pil_img.convert("RGB").crop((x1/1000*W, y1/1000*H,
                                            x2/1000*W, y2/1000*H))

    SYS = ("You are given the explanatory paragraph printed beneath a scientific "
           "figure. In ONE sentence, state what the figure shows and its key "
           "variables, to help a vision model caption it. No preamble.")

    _llm = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct",
                    model_kwargs={"quantization_config": BitsAndBytesConfig(
                        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)},
                    device_map="auto")

    def add_context(ex):
        ok, box = detect_text_block(ex["image"])
        if not ok:
            return {"context": ""}
        block = " ".join(_det.readtext(np.asarray(_crop_norm(ex["image"], box)),
                                       detail=0, paragraph=True)).strip()
        if not block:
            return {"context": ""}
        msg = [{"role": "system", "content": SYS},
               {"role": "user",   "content": block[:2000]}]
        out = _llm(msg, max_new_tokens=60, do_sample=False)[0]["generated_text"][-1]["content"]
        return {"context": out.strip()}

    grpo_raw = grpo_raw.map(add_context)
    del _llm; torch.cuda.empty_cache()
    print("context generated for",
          sum(bool(c) for c in grpo_raw["context"]), "/", len(grpo_raw), "figures")
else:
    grpo_raw = grpo_raw.add_column("context", [""] * len(grpo_raw))
    print("USE_CONTEXT=False -- prompts will carry no context prefix.")

USE_CONTEXT=False -- prompts will carry no context prefix.


In [ ]:
# Build the prompt column in the SFT format, and keep the reference caption
# around (as 'ref') so the eval cell later can use it. GRPO itself never sees
# 'ref' -- the reward is reference-free.
TASK = ("First localize each sub-figure as a JSON list of "
        "{\"bbox_2d\":[x1,y1,x2,y2], \"caption\":...}, coords normalized "
        "0-1000, then give an overall 'Summary:' of the whole figure.")

def build_prompt(ex):
    ctx = ex.get("context", "") or ""
    t = f"Context read from the figure: {ctx}\n\n{TASK}" if ctx else TASK
    return {"prompt": [{"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": t}]}],
            "ref": ex["caption"]}

grpo_dataset = grpo_raw.map(
    build_prompt,
    remove_columns=[c for c in grpo_raw.column_names if c != "image"])

print(grpo_dataset.column_names)                 # ['image', 'prompt', 'ref']
print(grpo_dataset[0]["prompt"][0]["role"])      # user
print(grpo_dataset[0]["prompt"][0]["content"][1]["text"][:160])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

['image', 'prompt', 'ref']
user
First localize each sub-figure as a JSON list of {"bbox_2d":[x1,y1,x2,y2], "caption":...}, coords normalized 0-1000, then give an overall 'Summary:' of the whol


---
## 6. Rewards

**`summary_clip_reward`** -- the cycle-consistency signal. The completion is
`Regions: [...]` followed by `Summary: ...`; only the summary span is a caption,
so only that is embedded against the figure. Falls back to the full completion
if no `Summary:` marker is emitted (early in training that happens a lot), with
a small penalty so the model learns to emit the marker.

**`format_reward`** -- shaping, capped at 0.25 so it never dominates the CLIP
term (which lands around 0.2-0.35). Rewards parseable region JSON with in-range
`bbox_2d` values and a non-trivial summary.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
import torch, json, re

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

def _text_of(completion):
    """GRPO hands back either a string or a chat-style list of blocks."""
    c = completion[0]["content"] if isinstance(completion, list) else completion
    if isinstance(c, list):
        c = " ".join(b.get("text", "") for b in c if isinstance(b, dict))
    return c or ""

def _split(text):
    """-> (regions_json_str, summary_str, had_marker)"""
    m = re.search(r"Summary\s*:", text)
    if not m:
        return text, text.strip(), False
    return text[:m.start()], text[m.end():].strip(), True

def summary_clip_reward(prompts, completions, **kwargs):
    images = kwargs.get("image", [])
    rewards = []
    for i, completion in enumerate(completions):
        text = _text_of(completion)
        _, summary, had_marker = _split(text)
        if not summary.strip():
            rewards.append(-1.0)          # empty output -> penalize, don't crash CLIP
            continue
        with torch.no_grad():
            inputs = clip_processor(
                text=[summary], images=[images[i]],
                return_tensors="pt", padding=True,
                truncation=True, max_length=77,      # CLIP hard limit
            ).to("cuda")
            sim = clip_model(**inputs).logits_per_image.squeeze().item()
        r = sim / 100.0
        if not had_marker:
            r -= 0.05                     # nudge toward emitting 'Summary:'
        rewards.append(r)
    return rewards

def format_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        text = _text_of(completion)
        regions_part, summary, had_marker = _split(text)
        r = 0.0
        m = re.search(r"\[.*\]", regions_part, re.S)
        if m:
            try:
                arr = json.loads(m.group(0))
                if isinstance(arr, list) and arr:
                    ok = all(
                        isinstance(d, dict)
                        and isinstance(d.get("bbox_2d"), list)
                        and len(d["bbox_2d"]) == 4
                        and all(isinstance(v, (int, float)) and 0 <= v <= 1000
                                for v in d["bbox_2d"])
                        for d in arr)
                    if ok:
                        r += 0.15
            except Exception:
                pass
        if had_marker and len(summary.split()) >= 5:
            r += 0.10
        rewards.append(r)                 # max 0.25
    return rewards

print("rewards ready")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

rewards ready


---
## 7. Hub login

In [ ]:
from huggingface_hub import login
login()   # paste a token with WRITE scope (or use Colab Secrets)

---
## 8. Train

`save_strategy="no"` means nothing is checkpointed to local disk; persistence is
entirely via periodic Hub pushes of the adapter (~200 MB), which is what keeps a
Colab disk from filling up. `remove_unused_columns=False` is **required** or the
`image` column never reaches the reward functions.

If you OOM: drop `num_generations` to 2 first, then `GRPO_LONGEST` to 768.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback
from unsloth import is_bf16_supported

class PushAdapterToHubCallback(TrainerCallback):
    """Persist by pushing the LoRA adapter to the Hub instead of writing
    multi-GB local checkpoints."""
    def __init__(self, repo, every):
        self.repo, self.every = repo, every
    def _push(self, tag):
        try:
            model.push_to_hub(self.repo, commit_message=tag)
            tokenizer.push_to_hub(self.repo, commit_message=tag)
            print(f"  ^ pushed adapter to {self.repo}  [{tag}]")
        except Exception as e:
            print(f"  ! Hub push failed [{tag}]: {e}  (training continues)")
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step > 0 and state.global_step % self.every == 0:
            self._push(f"step-{state.global_step}")
    def on_train_end(self, args, state, control, **kwargs):
        self._push("final")

training_args = GRPOConfig(
    output_dir = "outputs/grpo_dim_with_textlm",
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations = 4,            # drop to 2 if you OOM
    max_prompt_length = 2048,
    max_completion_length = 256,    # room for region JSON + summary
    learning_rate = 5e-6,
    num_train_epochs = 1,
    max_steps = MAX_STEPS,          # -1 => full epoch
    logging_steps = 1,
    optim = "adamw_8bit",
    bf16 = is_bf16_supported(),
    fp16 = not is_bf16_supported(),
    report_to = "none",
    save_strategy = "no",           # disk-safe; Hub is the checkpoint store
    remove_unused_columns = False,  # REQUIRED so 'image' reaches the reward fns
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [summary_clip_reward, format_reward],
    args = training_args,
    train_dataset = grpo_dataset,
    callbacks = [PushAdapterToHubCallback(GRPO_REPO, PUSH_EVERY)],
)

assert trainer.args.remove_unused_columns is False
print(f"GRPO starting from {SFT_ADAPTER}\n  pushing to {GRPO_REPO} every {PUSH_EVERY} steps")
trainer.train()
print("GRPO complete -- final adapter at:", GRPO_REPO)

GRPO starting from shemalfoy/qwen2-vl-scicap-lora-adapter-dim_with_textlm
  pushing to shemalfoy/qwen2-vl-scicap-grpo-dim_with_textlm every 50 steps


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 255
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 51,521,536 of 8,343,688,192 (0.62% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` wil

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / summary_clip_reward / mean,rewards / summary_clip_reward / std,rewards / format_reward / mean,rewards / format_reward / std
1,0.000749,0.534456,0.010988,116.250000,67.000000,196.000000,0.000000,116.250000,67.000000,196.000000,0.753224,0.284456,0.010988,0.250000,0.000000
2,0.000470,0.512476,0.005455,200.000000,152.000000,256.000000,0.250000,181.333344,152.000000,224.000000,0.477885,0.262476,0.005455,0.250000,0.000000
3,0.003558,0.538363,0.028350,109.750000,53.000000,153.000000,0.000000,109.750000,53.000000,153.000000,3.558581,0.288363,0.028350,0.250000,0.000000
4,0.000611,0.548052,0.014506,174.250000,133.000000,228.000000,0.000000,174.250000,133.000000,228.000000,0.611342,0.298052,0.014506,0.250000,0.000000
5,0.000623,0.537471,0.014124,133.500000,89.000000,202.000000,0.000000,133.500000,89.000000,202.000000,0.620983,0.287471,0.014124,0.250000,0.000000
6,0.000882,0.507185,0.018514,119.000000,56.000000,239.000000,0.000000,119.000000,56.000000,239.000000,0.882139,0.257185,0.018514,0.250000,0.000000
7,0.000722,0.549372,0.009799,133.750000,53.000000,232.000000,0.000000,133.750000,53.000000,232.000000,0.720113,0.299372,0.009799,0.250000,0.000000
8,0.001021,0.576782,0.019825,63.250000,54.000000,80.000000,0.000000,63.250000,54.000000,80.000000,1.020584,0.326782,0.019825,0.250000,0.000000
9,0.000950,0.561566,0.011264,111.750000,63.000000,196.000000,0.000000,111.750000,63.000000,196.000000,0.954255,0.311566,0.011264,0.250000,0.000000
10,0.000561,0.512678,0.021591,170.500000,99.000000,255.000000,0.000000,170.500000,99.000000,255.000000,0.561840,0.262678,0.021591,0.250000,0.000000


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpvkboofbr/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


  ^ pushed adapter to shemalfoy/qwen2-vl-scicap-grpo-dim_with_textlm  [final]
GRPO complete -- final adapter at: shemalfoy/qwen2-vl-scicap-grpo-dim_with_textlm


In [ ]:
# Belt-and-braces explicit push (the callback already does this on train_end,
# and also runs if you interrupt with the Stop button mid-run).
model.push_to_hub(GRPO_REPO)
tokenizer.push_to_hub(GRPO_REPO)
print("pushed ->", GRPO_REPO)

---
## 9. Resume (only if the run was interrupted)

Fresh runtime -> run cells 1, 2, 4, 5, 6, 7, then this instead of cell 8.
`max_steps` here is the number of **additional** steps.

In [ ]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
from unsloth import FastVisionModel, is_bf16_supported

model, tokenizer = FastVisionModel.from_pretrained(
    GRPO_REPO,                  # <- resume from the partially trained adapter
    max_seq_length = 16384,
    load_in_4bit   = True,
    fast_inference = False,
)
set_pixel_budget(tokenizer, 256*28*28, 1280*28*28)
FastVisionModel.for_training(model)
print("resumed from", GRPO_REPO)

---
## 10. Eval: SFT adapter vs GRPO adapter

Reference-based (BLEU / ROUGE-L against the ground-truth caption) **and**
reference-free (CLIP on the summary span, i.e. the training reward itself).
Both matter: GRPO optimizes CLIP directly, so CLIP going up is expected and is
not by itself evidence of a better caption. BLEU/ROUGE moving in the same
direction is the interesting result; CLIP up with ROUGE down means the policy is
drifting toward CLIP-pleasing text.

Held-out rows start after the GRPO slice, so neither stage has seen them.

In [ ]:
import torch, numpy as np, evaluate, gc
from PIL import Image as PILImage
from unsloth import FastVisionModel

N_EVAL   = 50
EVAL_FROM = GRPO_START + GRPO_N          # strictly after everything trained on
eval_df   = df.iloc[EVAL_FROM:EVAL_FROM + N_EVAL].reset_index(drop=True)
eval_ds   = [{"image": PILImage.open(r.image_path).convert("RGB"),
              "caption": str(r.caption)} for _, r in eval_df.iterrows()]
print(f"eval rows {EVAL_FROM}..{EVAL_FROM+len(eval_ds)-1}  ({len(eval_ds)} figures)")

rouge = evaluate.load("rouge")
bleu  = evaluate.load("bleu")

def _gen(m, tok, image):
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": TASK}]}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(image.convert("RGB"), text, add_special_tokens=False,
                 return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=256, do_sample=False,
                         use_cache=True)
    full = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    _, summary, _ = _split(full)
    return full, summary

def run(repo, label):
    m, tok = FastVisionModel.from_pretrained(repo, load_in_4bit=True,
                                             max_seq_length=16384,
                                             fast_inference=False)
    set_pixel_budget(tok, 256*28*28, 1280*28*28)
    FastVisionModel.for_inference(m)
    preds, clips = [], []
    for i, ex in enumerate(eval_ds):
        full, summary = _gen(m, tok, ex["image"])
        preds.append(summary if summary.strip() else full)
        with torch.no_grad():
            ci = clip_processor(text=[preds[-1]], images=[ex["image"]],
                                return_tensors="pt", padding=True,
                                truncation=True, max_length=77).to("cuda")
            clips.append(clip_model(**ci).logits_per_image.squeeze().item() / 100.0)
        if i % 10 == 0:
            print(f"  [{label}] {i}/{len(eval_ds)}  {preds[-1][:70]!r}")
    del m, tok; gc.collect(); torch.cuda.empty_cache()
    return preds, float(np.mean(clips))

refs = [ex["caption"] for ex in eval_ds]

sft_preds,  sft_clip  = run(SFT_ADAPTER, "SFT")
grpo_preds, grpo_clip = run(GRPO_REPO,   "GRPO")

def score(preds):
    r = rouge.compute(predictions=preds, references=refs)["rougeL"] * 100
    try:
        b = bleu.compute(predictions=preds, references=[[x] for x in refs])["bleu"] * 100
    except ZeroDivisionError:
        b = 0.0
    return b, r

sft_b,  sft_r  = score(sft_preds)
grpo_b, grpo_r = score(grpo_preds)

print(f"\n{'Metric':<12}{'SFT':>10}{'GRPO':>10}{'delta':>9}")
print("-" * 41)
for name, a, b in [("BLEU", sft_b, grpo_b),
                   ("ROUGE-L", sft_r, grpo_r),
                   ("CLIP", sft_clip * 100, grpo_clip * 100)]:
    arrow = "up" if b > a else ("down" if b < a else "--")
    print(f"{name:<12}{a:>10.2f}{b:>10.2f}{b-a:>+9.2f} {arrow}")

print("\n--- sample ---")
for i in range(min(3, len(eval_ds))):
    print(f"\n[{i}] REF : {refs[i][:180]}")
    print(f"    SFT : {sft_preds[i][:180]}")
    print(f"    GRPO: {grpo_preds[i][:180]}")

eval rows 2500..2549  (50 figures)


==((====))==  Unsloth 2026.8.21: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Unsloth: Offloading embeddings to RAM to save 1.02 GB.


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [GRPO] 30/50  '— The light curves in different energy bands for GRB 060904B. The inse'


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  [GRPO] 40/50  'The function kc(gA) for different values of the parameter α. The green'


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


Metric             SFT      GRPO    delta
-----------------------------------------
BLEU              5.28      6.28    +0.99 up
ROUGE-L          21.08     19.64    -1.44 down
CLIP             30.52     30.98    +0.47 up

--- sample ---

[0] REF : Cloud tag SPL keywords as example.
    SFT : The example of using Termine.
    GRPO: The example of the text and the result of the tool for seek relations in the text.

[1] REF : (Color online) Comparison between the numerical results of the effective permeability µ⊥r in a rectangular array for Λ/w > 0 (symbols) and the analytical results for Λ/w → 0 (lines
    SFT : (Color online) The ratio of the magnetic permeability in the left and right directions to that of the free space for the rectangular array with (a) different values of b/a and Λ/w 
    GRPO: (Color online) The ratio of the magnetic moment of the left and right spins in the rectangular array as a function of 1 − w/a for different values of b/a. The solid lines are the t

[2] REF 